# Appendix 03 — Advanced Atomics (companion, advanced)

> Independent **advanced** appendix, distilled from *CUDA by Example* (Sanders & Kandrot),
> **Appendix A — Advanced Atomics**. Closes the loop on Chapter 13b.
> Educational: `llm.c` never hand-rolls locks — and this appendix shows, on *your* GPU,
> exactly why that's the right call.

Chapter 13b used `atomicAdd`. But `atomicAdd`, `atomicMax`, and friends are all conveniences built on one universal primitive: **`atomicCAS`** (compare-and-swap). With `atomicCAS` you can build *any* atomic update. The book also builds a **mutex** from it and uses the mutex to finish the dot product's cross-block sum.

That second part is where this appendix departs from the book — honestly. The book was written for Fermi-era GPUs (2010). On **Volta and later** (compute ≥ 7.0, and this RTX 4080 SUPER is **8.9**), NVIDIA changed the execution model to *independent thread scheduling*, and the book's hand-rolled spinlock **deadlocks**. So we'll *show* the lock (it's instructive), explain precisely why it hangs on your hardware, and keep every **runnable** cell on the safe path: `atomicCAS` and `atomicAdd`.

### Learning objectives

By the end you will:

- Explain `atomicCAS(addr, compare, val)` and build a working **float `atomicAdd`** from it (runnable).
- Read a `atomicCAS`/`atomicExch` mutex and explain the classic **intra-warp deadlock**.
- Explain why even the "safe" one-thread-per-block lock **deadlocks on Volta+** — and why hardware atomics are the answer.


## 1. Concept — `atomicCAS`, the Universal Primitive

```c
int atomicCAS(int* addr, int compare, int val);
// atomically:  old = *addr;  if (old == compare) *addr = val;  return old;
```

It reads `*addr`, and *only if* it still equals `compare`, writes `val` — indivisibly — returning the original value. That "only if it hasn't changed since I last looked" is what lets you build a safe read-modify-write **retry loop**. Here's a float `atomicAdd` built entirely from integer CAS on the bit pattern:

```c
__device__ float atomicAddCAS(float* addr, float val) {
    int* ai = (int*)addr;
    int old = *ai, assumed;
    do {
        assumed = old;
        float updated = __int_as_float(assumed) + val;
        old = atomicCAS(ai, assumed, __float_as_int(updated));
    } while (assumed != old);     // someone changed it first -> retry with their value
    return __int_as_float(old);
}
```

Every atomic is either a hardware instruction or a CAS loop like this. We'll run this one and confirm it matches both the CPU and the hardware `atomicAdd`.


In [ ]:
!mkdir -p course/appendix03_build


In [ ]:
%%writefile course/appendix03_build/cas_add.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define imin(a,b) ((a)<(b)?(a):(b))
const int N               = 33 * 1024;
const int threadsPerBlock = 256;
const int blocksPerGrid   = imin(32, (N + threadsPerBlock - 1) / threadsPerBlock);

// float atomicAdd built from integer atomicCAS (the universal primitive)
__device__ float atomicAddCAS(float* addr, float val) {
    int* ai = (int*)addr;
    int old = *ai, assumed;
    do {
        assumed = old;
        float updated = __int_as_float(assumed) + val;
        old = atomicCAS(ai, assumed, __float_as_int(updated));
    } while (assumed != old);
    return __int_as_float(old);
}

// dot product: per-block shared reduction, then thread 0 CAS-adds into the global sum.
// No lock anywhere -> no deadlock. The CAS loop simply retries on contention.
__global__ void dot_cas(const float* a, const float* b, float* res) {
    __shared__ float cache[threadsPerBlock];
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    int c   = threadIdx.x;
    float temp = 0.0f;
    while (tid < N) { temp += a[tid] * b[tid]; tid += blockDim.x * gridDim.x; }
    cache[c] = temp;
    __syncthreads();
    for (int i = blockDim.x/2; i > 0; i /= 2) { if (c < i) cache[c] += cache[c+i]; __syncthreads(); }
    if (c == 0) atomicAddCAS(res, cache[0]);
}

int main(void) {
    float *a = (float*)malloc(N*4), *b = (float*)malloc(N*4);
    double cpu = 0.0;
    for (int i = 0; i < N; i++) { a[i] = i*1e-2f; b[i] = (i%7)*1e-2f; cpu += (double)a[i]*b[i]; }

    float *d_a, *d_b, *d_res;
    cudaMalloc(&d_a, N*4); cudaMalloc(&d_b, N*4); cudaMalloc(&d_res, 4);
    cudaMemcpy(d_a, a, N*4, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, N*4, cudaMemcpyHostToDevice);
    cudaMemset(d_res, 0, 4);

    dot_cas<<<blocksPerGrid, threadsPerBlock>>>(d_a, d_b, d_res);
    float gpu; cudaMemcpy(&gpu, d_res, 4, cudaMemcpyDeviceToHost);

    printf("CAS-built atomicAdd dot = %.4f\n", gpu);
    printf("CPU dot                 = %.4f\n", cpu);
    printf("rel diff = %.2e  -> %s\n", fabs(gpu - cpu)/fabs(cpu),
           (fabs(gpu - cpu)/fabs(cpu) < 1e-4) ? "PASS" : "FAIL");

    free(a); free(b);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_res);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/appendix03_build/cas_add course/appendix03_build/cas_add.cu && ./course/appendix03_build/cas_add


`PASS` — we just reimplemented `atomicAdd(float*)` from nothing but integer `atomicCAS`, and used it to combine the per-block partials on the GPU. The CAS loop handles contention by *retrying*, not by blocking — so it can never deadlock. (On modern hardware you'd of course just call the built-in `atomicAdd`; this shows what it's doing underneath.)


## 2. The Mutex — and Why It Deadlocks on Your GPU

The book goes further and builds a **mutex** from `atomicCAS`, then guards `*res += cache[0]` with it:

```c
struct Lock {
    int* mutex;
    __device__ void lock()   { while (atomicCAS(mutex, 0, 1) != 0); __threadfence(); }
    __device__ void unlock() { __threadfence(); atomicExch(mutex, 0); }
};
// ... if (threadIdx.x == 0) { lock.lock(); *res += cache[0]; lock.unlock(); }
```

There are **two** problems, one classic and one modern:

1. **Intra-warp deadlock (classic).** If two threads of the *same warp* call `lock()`, SIMT serializes the divergent branches: the thread holding the lock may never be scheduled to `unlock()` while its warp-mate spins. The book's mitigation is to let only **one thread per block** (`threadIdx.x == 0`) touch the lock, pushing contention to *across* blocks.

2. **Volta+ independent thread scheduling (modern).** Since compute 7.0, threads have independent program counters and there is **no guarantee** that a spinning thread ever yields to the lock-holder — even across blocks. On this RTX 4080 SUPER (8.9), the book's one-thread-per-block lock **hangs indefinitely**. *(This appendix originally tried to run it and had to be redesigned around exactly this deadlock.)*

That's the lesson, made concrete: **don't hand-roll locks on a GPU.** The CAS retry-loop in section 1 (and the hardware `atomicAdd` it mirrors) is *lock-free* — it makes progress under contention instead of waiting on a peer. We deliberately do **not** run the mutex code here, because on your hardware it would never return.


## 3. Translation Bridge

| Book (Appendix A) | Reality / `llm.c` | Takeaway |
|---|---|---|
| `atomicCAS` retry-loop | how a float `atomicAdd` works under the hood | CAS is the universal, **lock-free** primitive |
| hand-rolled `Lock` (mutex) | **never used**; deadlocks on Volta+ | GPU spinlocks are a footgun |
| one-thread-per-block lock | still hangs on compute ≥ 7.0 | independent thread scheduling broke the old trick |
| `__threadfence()` ordering | needed around any cross-block shared-state write | memory visibility isn't automatic |
| lock-based final sum | `atomicAdd` everywhere (e.g. `global_norm`) | prefer hardware atomics, always |

The throughline of Chapters 13a/13b and this appendix: combine work with **reductions** and **lock-free atomics**, never blocking locks. That's exactly what every kernel in `train_gpt2.cu` does.


## 4. Common Pitfalls

- **Hand-rolled spinlocks deadlock on Volta+** (compute ≥ 7.0) due to independent thread scheduling — the single biggest reason not to write them.
- **Intra-warp lock contention** deadlocks even on older GPUs.
- **A CAS loop must recompute from the *returned* old value** each iteration (`assumed = old`), or it will spin forever / corrupt data.
- **Missing `__threadfence()`** around cross-block shared-state updates → stale reads even when the atomic itself is correct.
- **Reaching for a lock where an atomic works** is slower, riskier, and — on your GPU — non-terminating.


## 5. TODO Exercise — Build `atomicMax` for floats from CAS

Implement a **lock-free** float `atomicMax` using the same `atomicCAS` retry pattern: keep trying to install `max(current, val)` until the swap succeeds. Fill in the TODO.


In [ ]:
%%writefile course/appendix03_build/exercise1.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define N 100000

__device__ float atomicMaxCAS(float* addr, float val) {
    int* ai = (int*)addr;
    int old = *ai, assumed;
    do {
        assumed = old;
        // TODO: compute the new bit pattern as float-max(assumed_as_float, val),
        //       then old = atomicCAS(ai, assumed, that_pattern);
        old = assumed;   // <-- placeholder: replace with the CAS attempt
    } while (assumed != old);
    return __int_as_float(old);
}

__global__ void reduce_max(const float* in, float* res, int n) {
    int i = threadIdx.x + blockIdx.x * blockDim.x;
    int stride = blockDim.x * gridDim.x;
    for (; i < n; i += stride) atomicMaxCAS(res, in[i]);
}

int main(void) {
    float* in = (float*)malloc(N*4);
    float truth = -1e30f;
    for (int i = 0; i < N; i++) { in[i] = sinf(i*0.3f)*1000.0f; if (in[i] > truth) truth = in[i]; }
    float *d_in, *d_res; cudaMalloc(&d_in, N*4); cudaMalloc(&d_res, 4);
    cudaMemcpy(d_in, in, N*4, cudaMemcpyHostToDevice);
    float neg = -1e30f; cudaMemcpy(d_res, &neg, 4, cudaMemcpyHostToDevice);
    reduce_max<<<64, 256>>>(d_in, d_res, N);
    float gpu; cudaMemcpy(&gpu, d_res, 4, cudaMemcpyDeviceToHost);
    printf("GPU max = %.4f  CPU max = %.4f  -> %s\n", gpu, truth,
           (fabs(gpu - truth) < 1e-2) ? "PASS" : "FAIL");
    free(in); cudaFree(d_in); cudaFree(d_res);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/appendix03_build/exercise1 course/appendix03_build/exercise1.cu && ./course/appendix03_build/exercise1


### Solution

In [ ]:
%%writefile course/appendix03_build/exercise1_sol.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define N 100000

__device__ float atomicMaxCAS(float* addr, float val) {
    int* ai = (int*)addr;
    int old = *ai, assumed;
    do {
        assumed = old;
        float m = fmaxf(__int_as_float(assumed), val);
        old = atomicCAS(ai, assumed, __float_as_int(m));
    } while (assumed != old);
    return __int_as_float(old);
}

__global__ void reduce_max(const float* in, float* res, int n) {
    int i = threadIdx.x + blockIdx.x * blockDim.x;
    int stride = blockDim.x * gridDim.x;
    for (; i < n; i += stride) atomicMaxCAS(res, in[i]);
}

int main(void) {
    float* in = (float*)malloc(N*4);
    float truth = -1e30f;
    for (int i = 0; i < N; i++) { in[i] = sinf(i*0.3f)*1000.0f; if (in[i] > truth) truth = in[i]; }
    float *d_in, *d_res; cudaMalloc(&d_in, N*4); cudaMalloc(&d_res, 4);
    cudaMemcpy(d_in, in, N*4, cudaMemcpyHostToDevice);
    float neg = -1e30f; cudaMemcpy(d_res, &neg, 4, cudaMemcpyHostToDevice);
    reduce_max<<<64, 256>>>(d_in, d_res, N);
    float gpu; cudaMemcpy(&gpu, d_res, 4, cudaMemcpyDeviceToHost);
    printf("GPU max = %.4f  CPU max = %.4f  -> %s\n", gpu, truth,
           (fabs(gpu - truth) < 1e-2) ? "PASS" : "FAIL");
    free(in); cudaFree(d_in); cudaFree(d_res);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/appendix03_build/exercise1_sol course/appendix03_build/exercise1_sol.cu && ./course/appendix03_build/exercise1_sol


## Recap

- **`atomicCAS`** is the universal atomic: every other atomic (add, max) is a hardware instruction or a lock-free CAS retry-loop on top of it.
- A CAS loop **makes progress under contention** by retrying — it can't deadlock.
- The book's `atomicCAS` **mutex deadlocks on Volta+** (compute ≥ 7.0, including this 4080) because of independent thread scheduling. We showed the code but did not run it.
- The practical rule: **reductions + lock-free atomics, never spinlocks.** `llm.c` follows it everywhere (`atomicAdd` in `global_norm`, `encoder_backward`).

### What's next

That completes the *CUDA by Example* companion set for the GPU half of the course:
**09a, 09b** (intro + parallel programming), **13a, 13b** (shared memory + atomics),
**Appendix 01–03** (constant memory/events, streams, advanced atomics), and **20a**
(zero-copy + multi-GPU). You've now built and run, by hand, the core CUDA primitives
behind `train_gpt2.cu`. 🎉
